# 🎬 Import et Analyse de Vidéos avec SportInsight AI

Ce notebook montre comment uploader et analyser vos propres vidéos de matchs de football.

## Installation des dépendances

In [ ]:
import sys
from pathlib import Path

# Ajouter le répertoire src au path
repo_root = Path.cwd()
if 'notebooks' in str(repo_root):
    repo_root = repo_root.parent
src_dir = repo_root / 'src'
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

print(f"Repo root: {repo_root}")
print(f"Src dir: {src_dir}")

In [ ]:
# Imports
import requests
import json
import time
from pathlib import Path
from IPython.display import display, HTML, Image, JSON
from IPython.core.display import clear_output
import pandas as pd

## Configuration de l'API

In [ ]:
# Configuration
API_BASE_URL = "http://localhost:8000"

# Vérifier que l'API est démarrée
try:
    response = requests.get(f"{API_BASE_URL}/health", timeout=5)
    print("✅ API est démarrée")
    print(response.json())
except requests.exceptions.ConnectionError:
    print("❌ API non accessible à", API_BASE_URL)
    print("Démarrez l'API avec: uvicorn apps.api.main:app --reload")

## Étape 1: Créer un nouveau job d'upload

In [ ]:
# Définir le nom du match
MATCH_NAME = "PSG vs Lyon"

# Créer le job
response = requests.post(
    f"{API_BASE_URL}/upload/create",
    data={"match_name": MATCH_NAME}
)

job_data = response.json()
JOB_ID = job_data["job_id"]

print(f"🆔 Job créé: {JOB_ID}")
print(f"📝 Match: {MATCH_NAME}")

## Étape 2: Upload des vidéos (Mi-temps 1 et 2)

In [ ]:
# Définir les chemins vers les vidéos
VIDEO_HALF_1 = "path/to/your/half1.mp4"  # À MODIFIER
VIDEO_HALF_2 = "path/to/your/half2.mp4"  # À MODIFIER

# Vérifier que les fichiers existent
for half, video_path in [(1, VIDEO_HALF_1), (2, VIDEO_HALF_2)]:
    if Path(video_path).exists():
        size_mb = Path(video_path).stat().st_size / 1024 / 1024
        print(f"✅ Vidéo {half}: {size_mb:.1f} MB")
    else:
        print(f"❌ Vidéo {half} non trouvée: {video_path}")

In [ ]:
# Upload mi-temps 1
print("📤 Upload mi-temps 1...")

with open(VIDEO_HALF_1, "rb") as f:
    files = {"file": f}
    response = requests.post(
        f"{API_BASE_URL}/upload/{JOB_ID}/video/1",
        files=files
    )

result = response.json()
print(f"✅ Mi-temps 1 uploadée: {result['size'] / 1024 / 1024:.1f} MB")

In [ ]:
# Upload mi-temps 2
print("📤 Upload mi-temps 2...")

with open(VIDEO_HALF_2, "rb") as f:
    files = {"file": f}
    response = requests.post(
        f"{API_BASE_URL}/upload/{JOB_ID}/video/2",
        files=files
    )

result = response.json()
print(f"✅ Mi-temps 2 uploadée: {result['size'] / 1024 / 1024:.1f} MB")

## Étape 3: Finaliser l'upload

In [ ]:
# Finaliser l'upload
print("🔒 Finalisation de l'upload...")

response = requests.post(f"{API_BASE_URL}/upload/{JOB_ID}/finalize")
status = response.json()

MATCH_DIR = status["match_dir"]

print(f"✅ Upload finalisé")
print(f"📁 Match directory: {MATCH_DIR}")
print(f"📊 Statut: {status['status']}")

## Étape 4: Extraire les features ResNET

⚠️ **Cette étape peut prendre 5-15 minutes** selon:
- La durée totale de la vidéo
- Le nombre de frames (dépend du paramètre fps)
- La puissance du GPU/CPU

In [ ]:
# Paramètres d'extraction
FPS = 2.0  # Frames par seconde à extraire (moins = plus rapide mais moins détaillé)
DEVICE = "auto"  # auto, cpu, ou cuda (GPU)

print(f"⚙️ Paramètres:")
print(f"  - FPS: {FPS}")
print(f"  - Device: {DEVICE}")
print()
print("🔄 Extraction des features (cela peut prendre du temps)...")
print()

response = requests.post(
    f"{API_BASE_URL}/upload/{JOB_ID}/extract-features",
    params={"fps": FPS, "device": DEVICE}
)

features_result = response.json()
print(f"✅ Features extraites")
print()

# Afficher les résultats par mi-temps
for half in [1, 2]:
    half_info = features_result["halves"][str(half)]
    if half_info["status"] == "success":
        print(f"✅ Mi-temps {half}:")
        print(f"   - Frames: {half_info['num_frames']}")
        print(f"   - Shape: {half_info['feature_shape']}")
        print(f"   - Features path: {half_info['features_path']}")
    else:
        print(f"❌ Mi-temps {half}: {half_info.get('message', 'Unknown error')}")

## Étape 5: Lancer l'inférence (Détection d'événements)

In [ ]:
# Récupérer la liste des checkpoints disponibles
response = requests.get(f"{API_BASE_URL}/checkpoints")
checkpoints = response.json()

print("📋 Checkpoints disponibles:")
for cp in checkpoints[:5]:
    print(f"  - {cp['name']} ({cp['id']})")

if checkpoints:
    CHECKPOINT = checkpoints[0]["path"]
    print(f"\n✅ Utilisation: {CHECKPOINT}")
else:
    CHECKPOINT = "runs/best.pt"  # À MODIFIER si pas de checkpoint trouvé
    print(f"\n⚠️ Aucun checkpoint trouvé. Utilisation par défaut: {CHECKPOINT}")

In [ ]:
# Lancer l'inférence
print("🎬 Lancement de l'inférence...")
print()

response = requests.post(
    f"{API_BASE_URL}/inference/run",
    json={
        "match_dir": MATCH_DIR,
        "checkpoint": CHECKPOINT,
        "half": "both",
        "score_threshold": 0.30,
        "nms_radius_sec": 6.0,
    }
)

inference_result = response.json()

print(f"✅ Inférence terminée")
print()

# Afficher le résumé
summary = inference_result["summary"]
print(f"📊 Résumé:")
print(f"  - Événements détectés: {summary['event_count']}")
print(f"  - Run ID: {summary['run_id']}")
print()

## Étape 6: Afficher les résultats

In [ ]:
# Extraire les événements
events = inference_result["events"]
summary = inference_result["summary"]

# Convertir en DataFrame pour une meilleure visualisation
events_df = pd.DataFrame(events)

# Afficher les premières lignes
print("🎬 Événements détectés:")
print()
print(events_df[["half", "gameTime", "label", "score"]].head(20).to_string(index=False))

if len(events_df) > 20:
    print(f"\n... et {len(events_df) - 20} autres événements")

In [ ]:
# Statistiques par type d'événement
print("\n📊 Événements par classe:")
print()

counts_by_class = summary["counts_by_class"]
for label, count in sorted(counts_by_class.items(), key=lambda x: x[1], reverse=True):
    print(f"  {label}: {count}")

In [ ]:
# Événements par mi-temps
print("\n📊 Événements par mi-temps:")
print()

events_by_half = events_df.groupby("half").size()
for half, count in events_by_half.items():
    print(f"  Mi-temps {half}: {count} événements")

In [ ]:
# Visualisation: Confiance moyenne par classe
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Nombre d'événements par classe
class_counts = events_df["label"].value_counts()
ax1.barh(class_counts.index, class_counts.values, color="steelblue")
ax1.set_xlabel("Nombre d'événements")
ax1.set_title("Événements par classe")
ax1.grid(axis="x", alpha=0.3)

# Confiance moyenne par classe
avg_score = events_df.groupby("label")["score"].mean()
ax2.barh(avg_score.index, avg_score.values, color="coral")
ax2.set_xlabel("Confiance moyenne")
ax2.set_title("Confiance moyenne par classe")
ax2.set_xlim(0, 1)
ax2.grid(axis="x", alpha=0.3)

plt.tight_layout()
plt.show()

## Étape 7: Regarder des clips autour des événements

In [ ]:
# Sélectionner un événement avec confiance élevée
high_confidence_events = events_df[events_df["score"] > 0.5].head(1)

if len(high_confidence_events) > 0:
    event = high_confidence_events.iloc[0]
    print(f"🎬 Événement sélectionné:")
    print(f"  - Type: {event['label']}")
    print(f"  - Time: {event['gameTime']}")
    print(f"  - Confiance: {event['score']:.2%}")
    print(f"  - Mi-temps: {event['half']}")
    print()
    print("💡 Génération d'un clip autour de cet événement...")
else:
    print("Aucun événement avec confiance > 0.5 trouvé")

In [ ]:
# Générer un clip si ffmpeg est disponible
if len(high_confidence_events) > 0:
    event = high_confidence_events.iloc[0]
    
    try:
        response = requests.get(
            f"{API_BASE_URL}/media/clip",
            params={
                "match_dir": MATCH_DIR,
                "half": int(event["half"]),
                "timestamp": float(event["timestamp"]),
                "before_sec": 10.0,
                "after_sec": 10.0,
            },
            timeout=60
        )
        
        if response.status_code == 200:
            # Sauvegarder le clip
            clip_path = Path("event_clip.mp4")
            with open(clip_path, "wb") as f:
                f.write(response.content)
            print(f"✅ Clip généré: {clip_path}")
        else:
            print(f"⚠️ Erreur: {response.json()['detail']}")
    except Exception as e:
        print(f"⚠️ Impossible de générer le clip: {e}")

## Nettoyage (Optionnel)

In [ ]:
# Supprimer le job (optionnel)
# Cela supprimera tous les fichiers uploadés et extraits

# response = requests.delete(f"{API_BASE_URL}/upload/{JOB_ID}")
# print(response.json())